In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import json

from functools import partial
from pathlib import Path

import pandas as pd
import scanpy as sc
import numpy as np

from requests import post
from dotenv import dotenv_values

In [3]:
# Load config file agnostic to project or notebook work directory
config_path = Path("config.env")
if not config_path.exists():
    config_path = Path("../../../config.env")
CONFIG = dotenv_values(config_path)
assert CONFIG != {}, 'Configuration is empty - check the filepath.'

# Main data directory
DATADIR = Path(CONFIG['DATADIR'])

# Submodule specific data directory
OUTDIR = DATADIR / '10_analyses' / 'sexual_dimorphism'
OUTDIR.mkdir(parents=True, exist_ok=True)

# Notebook-specific constants
SIG_THRESHOLD = 0.05
N_SIG_CLUSTER_THRESHOLD = 10
PCT_EXPR_THRESHOLD = 0.1

In [4]:
# Read DEG results from disk
df_sex_degs = pd.read_csv(OUTDIR / 'L4_multivariate_sex_edgeR_results_full.csv', index_col=0)
df_sex_degs.head(3)

,logFC,logCPM,F,PValue,FDR,id
Xist,-9.366312,9.111565,693.382285,1.626745e-23,6.650134e-20,Gl4-293
Uty,6.822763,6.542082,398.997189,6.205220e-22,1.268347e-18,Gl4-293
Eif2s3y,7.544865,6.604348,321.044229,2.380016e-16,3.243169e-13,Gl4-293


In [10]:
# Map for gene names to ids
df_gene_metadata = pd.read_csv(DATADIR / 'gene_metadata.csv', index_col='gene_name')
dict_name_to_id = df_gene_metadata['gene_ids'].to_dict()

### Setup g:profiler API call architecture

In [5]:
TEMPLATE_QUERY = {
    'query': 'REPLACE ME',
    'background': 'REPLACE ME',
    'organism': 'mmusculus',
    'sources': ['GO:MF', 'GO:BP'], #only look into Gene Ontology terms.
    'significance_threshold_method': 'g_SCS', #use bonferroni correction instrad of the default 'g_SCS'.
    'no_evidences': True, #skip lookup for evidence codes. Speeds up queries, if there is no interest in evidence codes.
    'no_iea': False, #Ignore electonically annotated GO annotations
    'domain_scope': 'custom', #use the genes in the probe as the statistical background.
    'highlight': True
}

partial_post = partial(
    post,
    url='https://biit.cs.ut.ee/gprofiler/api/gost/profile/',
    headers={
        'User-Agent':'FullPythonRequest'
    }
)

### Global DEG pathways

In [17]:
# Extract possible universe (background) - all tested genes
universe = df_sex_degs.index.unique().tolist()

# Extract genes that are signficant 10 times across all celltypes
df_sig_sex_degs = df_sex_degs[df_sex_degs['FDR'] < SIG_THRESHOLD]
gene_sig_counts = df_sig_sex_degs.groupby(df_sig_sex_degs.index, observed=True).size()
query_genes = gene_sig_counts[gene_sig_counts >= N_SIG_CLUSTER_THRESHOLD].index.tolist()

print(len(query_genes), len(universe))

166 15685


In [18]:
# Conert from gene_names to IDs for gprofiler
query_genes = [dict_name_to_id[gene] for gene in query_genes]
universe = [dict_name_to_id[gene] for gene in universe]

In [19]:
# Create output folder
result_outdir = OUTDIR / 'GO_results' / 'global'
result_outdir.mkdir(parents=True, exist_ok=True)

# Setup query content
search_query = TEMPLATE_QUERY.copy()
search_query['query'] = query_genes
search_query['background'] = universe

# Submit via API
r = partial_post(
    json=search_query
)
go_results, post_metadata = r.json().values()

# Save out metadata to disk as a record of API version etc
outfile = result_outdir / 'metadata.json'
with open(outfile, 'w') as fp:
    json.dump(post_metadata, fp, indent=4)

# Remove parents field as it is a list (unsupported by pd.DataFrame)
go_results = [
    {k: v for k, v in result.items() if k != 'parents'}
    for result in go_results
]

# Convert to DataFrame
df_go_results = pd.DataFrame(go_results)
print(f"{df_go_results.__len__()} GO terms found!")

# Save out DataFrame to disk
outfile = result_outdir / 'results.csv'
df_go_results.to_csv(outfile)

26 GO terms found!


### Cell-type GO pathways

In [8]:
# Read full control anndata from disk
fp_adata = DATADIR / '9_objects' / 'singlecell' / 'hypomap3d_singlecell_control.h5ad'
adata_sc = sc.read_h5ad(fp_adata)

In [9]:
# Precompute boolean
X_bool = adata_sc.X > 0
X_bool

<Compressed Sparse Row sparse matrix of dtype 'bool'
	with 2694441755 stored elements and shape (896959, 56939)>

In [10]:
# Manually select cell-types to be tested
CELLTYPE_TO_RUN = [
    'Gl4-213',
    'Gl4-220',
    'Gb4-444'
]

In [11]:
for celltype in CELLTYPE_TO_RUN:
   
    # Create mask
    X_mask = adata_sc.obs["L4_ID"] == celltype
    
    # Sanity check
    assert np.any(X_mask), f'{celltype} was not found at the specified level'
    
    # Subset to only cells of that celltype and compute percentage of cells expressing each gene
    sX_bool = X_bool[X_mask, :]
    pct_gene_expressed = np.asarray(sX_bool.mean(axis=0)).flatten()
    
    universe = adata_sc.var_names[pct_gene_expressed > PCT_EXPR_THRESHOLD].tolist()

    assert celltype in df_sex_degs['id'].values, f"{celltype} does not appear in DEG results - it may not have been tested..."
    
    df_mask = df_sex_degs['id'] == celltype
    subset_df_sex_degs = df_sex_degs.loc[df_mask]
    
    # Subet to only significant genes
    subset_df_sex_degs = df_sex_degs.loc[df_mask]
    subset_df_sig_sex_degs = subset_df_sex_degs[subset_df_sex_degs['FDR'] < SIG_THRESHOLD]
    
    print(len(subset_df_sig_sex_degs), 'sex DEGs for', celltype)
    
    query = subset_df_sig_sex_degs.index.tolist()
    
    print(len(query), len(universe))
    
    if len(query) < 100:
        print(f"Less than 100 genes are present in query [{len(query)}] - results likely to be poor.")
    
    # Create output folder
    result_outdir = OUTDIR / 'GO_results' / celltype
    result_outdir.mkdir(parents=True, exist_ok=True)

    # Setup query content
    search_query = TEMPLATE_QUERY.copy()
    search_query['query'] = query
    search_query['background'] = universe

    # Submit via API
    r = partial_post(
        json=search_query
    )
    go_results, post_metadata = r.json().values()

    # Save out metadata to disk as a record of API version etc
    outfile = result_outdir / 'metadata.json'
    with open(outfile, 'w') as fp:
        json.dump(post_metadata, fp, indent=4)

    # Remove parents field as it is a list (unsupported by pd.DataFrame)
    go_results = [
        {k: v for k, v in result.items() if k != 'parents'}
        for result in go_results
    ]

    # Convert to DataFrame
    df_go_results = pd.DataFrame(go_results)
    print(f"{df_go_results.__len__()} GO terms found!")

    # Save out DataFrame to disk
    outfile = result_outdir / 'results.csv'
    df_go_results.to_csv(outfile)

400 sex DEGs for Gl4-213
400 9738
34 GO terms found!
561 sex DEGs for Gl4-220
561 10976
73 GO terms found!
264 sex DEGs for Gb4-444
264 9543
11 GO terms found!


### Compile supplementary table

In [19]:
fp_go_result_dirs = sorted((OUTDIR / "GO_results").glob("*/results.csv"))

df_compiled = (
    pd.concat(
        (
            pd.read_csv(fp, index_col=0)
            for fp in fp_go_result_dirs
        ),
        keys=[fp.parent.stem for fp in fp_go_result_dirs],
        names=["target"],
    )
    .reset_index(level=0)
    .reset_index(drop=True)
)

outfile = OUTDIR / 'GO_results' / 'compiled_table.csv'
df_compiled.to_csv(outfile, index=None)